In [1]:
# ============================================================
# CELL 1 NOTEBOOK 8 — FINAL MODEL EVALUATION & COMPARISON
# ============================================================
import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import joblib

from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("NOTEBOOK 8 — FINAL MODEL EVALUATION & COMPARISON")
print("=" * 70)

print(f"XGBoost version: {xgb.__version__}")
print(f"Optuna version:  {optuna.__version__}")

#Cell 2 — Define project paths/directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


NOTEBOOK 8 — FINAL MODEL EVALUATION & COMPARISON
XGBoost version: 3.3.0
Optuna version:  4.9.0


In [2]:
#Cell 3 — Load the untouched test dataset
#We will use the original 117-row test set containing the 61 modelling predictors.
# ============================================================
# LOAD ORIGINAL TEST DATA
# ============================================================

test_binary_processed = pd.read_csv(
    PROCESSED_DIR / "test_binary_processed.csv"
)

TARGET = "RISK_BINARY"

X_test_full = test_binary_processed.drop(
    columns=[TARGET, "RISK_LABEL"],
    errors="ignore"
)

y_test = test_binary_processed[TARGET].copy()

print("=" * 70)
print("ORIGINAL TEST DATA LOADED")
print("=" * 70)

print(f"Test observations: {len(y_test)}")
print(f"Test predictors:   {X_test_full.shape[1]}")

assert len(y_test) == 117
assert X_test_full.shape == (117, 61)

print("\n✓ Original test set confirmed: 117 × 61")

ORIGINAL TEST DATA LOADED
Test observations: 117
Test predictors:   61

✓ Original test set confirmed: 117 × 61


In [3]:
# ============================================================
#CELL 4- LOAD SHAP-PRUNED TEST DATA
# ============================================================

test_binary_shap = pd.read_csv(
    PROCESSED_DIR / "test_binary_shap.csv"
)

X_test_shap = test_binary_shap.drop(
    columns=[TARGET]
)

y_test_shap = test_binary_shap[TARGET].copy()

print("=" * 70)
print("SHAP TEST DATA LOADED")
print("=" * 70)

print(f"Test observations: {len(y_test_shap)}")
print(f"SHAP predictors:   {X_test_shap.shape[1]}")

assert X_test_shap.shape == (117, 49)
assert y_test_shap.equals(y_test)

print("\n✓ SHAP test set confirmed: 117 × 49")
print("✓ Target alignment confirmed")

SHAP TEST DATA LOADED
Test observations: 117
SHAP predictors:   49

✓ SHAP test set confirmed: 117 × 49
✓ Target alignment confirmed


In [4]:
#Cell 5 — Load the three trained models
# ============================================================
# LOAD TRAINED MODELS
# ============================================================

baseline_model_path = (
    MODELS_DIR / "baseline_xgboost_binary.pkl"
)

s_model_path = (
    MODELS_DIR / "s_xgboost_binary.pkl"
)

sp_model_path = (
    MODELS_DIR / "sp_xgboost_binary.pkl"
)

print("=" * 70)
print("MODEL FILE VALIDATION")
print("=" * 70)

print(f"Baseline:   {baseline_model_path}")
print(f"S-XGBoost:  {s_model_path}")
print(f"SP-XGBoost: {sp_model_path}")

assert baseline_model_path.exists()
assert s_model_path.exists()
assert sp_model_path.exists()

baseline_model = joblib.load(
    baseline_model_path
)

s_model = joblib.load(
    s_model_path
)

sp_model = joblib.load(
    sp_model_path
)

print("\n✓ All three trained models loaded")

MODEL FILE VALIDATION
Baseline:   ..\models\baseline_xgboost_binary.pkl
S-XGBoost:  ..\models\s_xgboost_binary.pkl
SP-XGBoost: ..\models\sp_xgboost_binary.pkl

✓ All three trained models loaded


In [5]:
#Cell 6 — Validate model feature spaces, 
# This is a critical validation cell.
# ============================================================
# MODEL / FEATURE ALIGNMENT VALIDATION
# ============================================================

print("=" * 70)
print("MODEL / FEATURE ALIGNMENT VALIDATION")
print("=" * 70)

baseline_features = list(
    baseline_model.get_booster().feature_names
)

s_features = list(
    s_model.get_booster().feature_names
)

sp_features = list(
    sp_model.get_booster().feature_names
)

print(f"Baseline model features:  {len(baseline_features)}")
print(f"S-XGBoost model features: {len(s_features)}")
print(f"SP-XGBoost model features:{len(sp_features)}")

assert len(baseline_features) == 61
assert len(s_features) == 49
assert len(sp_features) == 49

assert baseline_features == list(
    X_test_full.columns
)

assert s_features == list(
    X_test_shap.columns
)

assert sp_features == list(
    X_test_shap.columns
)

print("\n✓ Baseline model matches 61-feature test space")
print("✓ S-XGBoost matches 49-feature test space")
print("✓ SP-XGBoost matches 49-feature test space")
print("✓ Feature order validated")


MODEL / FEATURE ALIGNMENT VALIDATION
Baseline model features:  61
S-XGBoost model features: 49
SP-XGBoost model features:49

✓ Baseline model matches 61-feature test space
✓ S-XGBoost matches 49-feature test space
✓ SP-XGBoost matches 49-feature test space
✓ Feature order validated


In [6]:
# ============================================================
# CELL 7- GENERATE FINAL TEST PREDICTIONS
# ============================================================

# Baseline
baseline_prob = baseline_model.predict_proba(
    X_test_full
)[:, 1]

baseline_pred = (
    baseline_prob >= 0.50
).astype(int)


# S-XGBoost
s_prob = s_model.predict_proba(
    X_test_shap
)[:, 1]

s_pred = (
    s_prob >= 0.50
).astype(int)


# SP-XGBoost
sp_prob = sp_model.predict_proba(
    X_test_shap
)[:, 1]

sp_pred = (
    sp_prob >= 0.50
).astype(int)


print("=" * 70)
print("FINAL TEST PREDICTIONS GENERATED")
print("=" * 70)

print(f"Baseline predictions:   {len(baseline_pred)}")
print(f"S-XGBoost predictions:  {len(s_pred)}")
print(f"SP-XGBoost predictions:{len(sp_pred)}")

assert len(baseline_pred) == 117
assert len(s_pred) == 117
assert len(sp_pred) == 117

assert len(baseline_prob) == 117
assert len(s_prob) == 117
assert len(sp_prob) == 117

print("\n✓ All three models evaluated on the same 117 observations")

FINAL TEST PREDICTIONS GENERATED
Baseline predictions:   117
S-XGBoost predictions:  117
SP-XGBoost predictions:117

✓ All three models evaluated on the same 117 observations


In [7]:
#Cell 8 — Validate probabilities
# ============================================================
# PROBABILITY VALIDATION
# ============================================================

print("=" * 70)
print("PROBABILITY VALIDATION")
print("=" * 70)

for name, probabilities in {
    "Baseline XGBoost": baseline_prob,
    "S-XGBoost": s_prob,
    "SP-XGBoost": sp_prob
}.items():

    assert np.all(
        (probabilities >= 0) &
        (probabilities <= 1)
    )

    assert np.all(
        np.isfinite(probabilities)
    )

    print(f"✓ {name}: probabilities valid")

print("\n✓ ALL PROBABILITIES VALID")

PROBABILITY VALIDATION
✓ Baseline XGBoost: probabilities valid
✓ S-XGBoost: probabilities valid
✓ SP-XGBoost: probabilities valid

✓ ALL PROBABILITIES VALID


In [8]:
#Cell 9 — Evaluation function
#This keeps the metric calculation identical for all three models.
# ============================================================
# MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_binary_model(
    y_true,
    y_pred,
    y_prob
):

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    return {
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "Accuracy": accuracy,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

print("✓ Evaluation function defined")

✓ Evaluation function defined


In [9]:
#Cell 10 — Calculate final metrics
# ============================================================
# FINAL TEST-SET METRICS
# ============================================================

baseline_metrics = evaluate_binary_model(
    y_test,
    baseline_pred,
    baseline_prob
)

s_metrics = evaluate_binary_model(
    y_test,
    s_pred,
    s_prob
)

sp_metrics = evaluate_binary_model(
    y_test,
    sp_pred,
    sp_prob
)

print("=" * 70)
print("FINAL TEST-SET METRICS CALCULATED")
print("=" * 70)

print("✓ Baseline XGBoost")
print("✓ S-XGBoost")
print("✓ SP-XGBoost")

print("\n✓ All metrics calculated on the same 117 test observations")

FINAL TEST-SET METRICS CALCULATED
✓ Baseline XGBoost
✓ S-XGBoost
✓ SP-XGBoost

✓ All metrics calculated on the same 117 test observations


In [10]:
#Cell 11 — Display the final comparison
# ============================================================
# FINAL THREE-MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "Baseline XGBoost",
        "S-XGBoost",
        "SP-XGBoost"
    ],

    "Predictors": [
        61,
        49,
        49
    ],

    "ROC-AUC": [
        baseline_metrics["ROC_AUC"],
        s_metrics["ROC_AUC"],
        sp_metrics["ROC_AUC"]
    ],

    "PR-AUC": [
        baseline_metrics["PR_AUC"],
        s_metrics["PR_AUC"],
        sp_metrics["PR_AUC"]
    ],

    "F1": [
        baseline_metrics["F1"],
        s_metrics["F1"],
        sp_metrics["F1"]
    ],

    "Precision": [
        baseline_metrics["Precision"],
        s_metrics["Precision"],
        sp_metrics["Precision"]
    ],

    "Recall": [
        baseline_metrics["Recall"],
        s_metrics["Recall"],
        sp_metrics["Recall"]
    ],

    "Specificity": [
        baseline_metrics["Specificity"],
        s_metrics["Specificity"],
        sp_metrics["Specificity"]
    ],

    "Accuracy": [
        baseline_metrics["Accuracy"],
        s_metrics["Accuracy"],
        sp_metrics["Accuracy"]
    ]
})

print("=" * 70)
print("FINAL MODEL COMPARISON — TEST SET")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

FINAL MODEL COMPARISON — TEST SET
           Model  Predictors  ROC-AUC  PR-AUC     F1  Precision  Recall  Specificity  Accuracy
Baseline XGBoost          61   0.7567  0.7220 0.6296     0.6538  0.6071       0.7049    0.6581
       S-XGBoost          49   0.7681  0.7395 0.6476     0.6939  0.6071       0.7541    0.6838
      SP-XGBoost          49   0.8138  0.7751 0.7059     0.7826  0.6429       0.8361    0.7436


In [11]:
#Cell 12 — Confusion matrices
# ============================================================
# CONFUSION MATRICES
# ============================================================

print("=" * 70)
print("CONFUSION MATRICES")
print("=" * 70)

print("\nBASELINE XGBOOST")
print(baseline_metrics["TN"],
      baseline_metrics["FP"])
print(baseline_metrics["FN"],
      baseline_metrics["TP"])

print("\nS-XGBOOST")
print(s_metrics["TN"],
      s_metrics["FP"])
print(s_metrics["FN"],
      s_metrics["TP"])

print("\nSP-XGBOOST")
print(sp_metrics["TN"],
      sp_metrics["FP"])
print(sp_metrics["FN"],
      sp_metrics["TP"])

CONFUSION MATRICES

BASELINE XGBOOST
43 18
22 34

S-XGBOOST
46 15
22 34

SP-XGBOOST
51 10
20 36


In [12]:
#Cell 13 — Performance changes, important for your ablation study.
# ============================================================
# PERFORMANCE CHANGE ANALYSIS
# ============================================================

metrics_to_compare = [
    "ROC-AUC",
    "PR-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy"
]

improvement = pd.DataFrame({
    "Metric": metrics_to_compare,

    "S_minus_Baseline": [
        comparison.loc[1, metric] -
        comparison.loc[0, metric]
        for metric in metrics_to_compare
    ],

    "SP_minus_S": [
        comparison.loc[2, metric] -
        comparison.loc[1, metric]
        for metric in metrics_to_compare
    ],

    "SP_minus_Baseline": [
        comparison.loc[2, metric] -
        comparison.loc[0, metric]
        for metric in metrics_to_compare
    ]
})

print("=" * 70)
print("PERFORMANCE CHANGE ANALYSIS")
print("=" * 70)

print(
    improvement.to_string(
        index=False,
        float_format=lambda x: f"{x:+.4f}"
    )
)

PERFORMANCE CHANGE ANALYSIS
     Metric  S_minus_Baseline  SP_minus_S  SP_minus_Baseline
    ROC-AUC           +0.0114     +0.0457            +0.0571
     PR-AUC           +0.0175     +0.0357            +0.0532
         F1           +0.0180     +0.0583            +0.0763
  Precision           +0.0400     +0.0887            +0.1288
     Recall           +0.0000     +0.0357            +0.0357
Specificity           +0.0492     +0.0820            +0.1311
   Accuracy           +0.0256     +0.0598            +0.0855


In [13]:
#Cell 14 — Determine the best model
#Do not select the model using only accuracy. For your early-warning problem, 
# ROC-AUC and PR-AUC are particularly important, 
# while recall is also important because failing to identify an at-risk student is consequential.
# ============================================================
# BEST MODEL SUMMARY
# ============================================================

print("=" * 70)
print("BEST MODEL SUMMARY")
print("=" * 70)

best_auc_idx = comparison["ROC-AUC"].idxmax()
best_prauc_idx = comparison["PR-AUC"].idxmax()
best_f1_idx = comparison["F1"].idxmax()
best_recall_idx = comparison["Recall"].idxmax()

print(
    f"Best ROC-AUC:   {comparison.loc[best_auc_idx, 'Model']}"
)

print(
    f"Best PR-AUC:    {comparison.loc[best_prauc_idx, 'Model']}"
)

print(
    f"Best F1:        {comparison.loc[best_f1_idx, 'Model']}"
)

print(
    f"Best Recall:    {comparison.loc[best_recall_idx, 'Model']}"
)

BEST MODEL SUMMARY
Best ROC-AUC:   SP-XGBoost
Best PR-AUC:    SP-XGBoost
Best F1:        SP-XGBoost
Best Recall:    SP-XGBoost


In [14]:
# ============================================================
# CELL 15 -SAVE FINAL EVALUATION RESULTS
# ============================================================

comparison_path = (
    RESULTS_DIR / "final_model_comparison.csv"
)

improvement_path = (
    RESULTS_DIR / "model_performance_changes.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

improvement.to_csv(
    improvement_path,
    index=False
)

print("=" * 70)
print("FINAL RESULTS SAVED")
print("=" * 70)

print(f"Model comparison: {comparison_path}")
print(f"Performance changes: {improvement_path}")

assert comparison_path.exists()
assert improvement_path.exists()

print("\n✓ Final evaluation results saved")

FINAL RESULTS SAVED
Model comparison: ..\results\final_model_comparison.csv
Performance changes: ..\results\model_performance_changes.csv

✓ Final evaluation results saved


In [15]:
#Cell 16 — Save predictions, This is useful for reproducibility, 
# and later statistical/error analysis.
# ============================================================
# SAVE TEST-SET PREDICTIONS
# ============================================================

predictions = pd.DataFrame({
    "RISK_BINARY_ACTUAL": y_test.values,

    "Baseline_Probability": baseline_prob,
    "Baseline_Prediction": baseline_pred,

    "S_XGBoost_Probability": s_prob,
    "S_XGBoost_Prediction": s_pred,

    "SP_XGBoost_Probability": sp_prob,
    "SP_XGBoost_Prediction": sp_pred
})

predictions_path = (
    RESULTS_DIR / "final_test_predictions.csv"
)

predictions.to_csv(
    predictions_path,
    index=False
)

print("=" * 70)
print("TEST PREDICTIONS SAVED")
print("=" * 70)

print(f"File: {predictions_path}")

assert predictions.shape == (117, 7)
assert predictions_path.exists()

print("✓ 117 test observations saved")

TEST PREDICTIONS SAVED
File: ..\results\final_test_predictions.csv
✓ 117 test observations saved


In [16]:
# ============================================================
# CELL 17 - NOTEBOOK 8 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 8 — FINAL VALIDATION")
print("=" * 70)

# Test size
assert len(y_test) == 117

# Feature spaces
assert X_test_full.shape == (117, 61)
assert X_test_shap.shape == (117, 49)

# Predictions
assert len(baseline_pred) == 117
assert len(s_pred) == 117
assert len(sp_pred) == 117

# Probabilities
for probabilities in [
    baseline_prob,
    s_prob,
    sp_prob
]:

    assert len(probabilities) == 117
    assert np.all(
        (probabilities >= 0) &
        (probabilities <= 1)
    )

# Metrics
for metrics in [
    baseline_metrics,
    s_metrics,
    sp_metrics
]:

    for key in [
        "ROC_AUC",
        "PR_AUC",
        "F1",
        "Precision",
        "Recall",
        "Specificity",
        "Accuracy"
    ]:

        assert np.isfinite(
            metrics[key]
        )

# Confusion matrices
assert (
    baseline_metrics["TN"] +
    baseline_metrics["FP"] +
    baseline_metrics["FN"] +
    baseline_metrics["TP"]
) == 117

assert (
    s_metrics["TN"] +
    s_metrics["FP"] +
    s_metrics["FN"] +
    s_metrics["TP"]
) == 117

assert (
    sp_metrics["TN"] +
    sp_metrics["FP"] +
    sp_metrics["FN"] +
    sp_metrics["TP"]
) == 117

print("✓ Same 117 test observations used for all models")
print("✓ Baseline: 61 predictors")
print("✓ S-XGBoost: 49 predictors")
print("✓ SP-XGBoost: 49 predictors")
print("✓ Predictions valid")
print("✓ Probabilities valid")
print("✓ All evaluation metrics finite")
print("✓ Confusion matrices valid")
print("✓ Final comparison saved")
print("✓ Test set remains exactly 117 observations")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 8 FINAL VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 8 — FINAL VALIDATION
✓ Same 117 test observations used for all models
✓ Baseline: 61 predictors
✓ S-XGBoost: 49 predictors
✓ SP-XGBoost: 49 predictors
✓ Predictions valid
✓ Probabilities valid
✓ All evaluation metrics finite
✓ Confusion matrices valid
✓ Final comparison saved
✓ Test set remains exactly 117 observations

✓ NOTEBOOK 8 FINAL VALIDATION PASSED
